# v3 — Convolutional Autoencoder (small-sample teaching)

Follows the **semi-supervised Conv-AE** in Lyu et al. (ChemRxiv 2026, *Benchmarking Machine Learning Fault Detection Methods on the Tennessee Eastman Process Dataset*):

> The Convolutional Autoencoder uses 1D convolutional layers for both encoding and decoding. The encoder applies strided convolutions to progressively reduce the temporal dimension while increasing feature channels, compressing the input into a latent representation. The decoder uses transposed convolutions to reconstruct the original sequence. This architecture efficiently captures local temporal patterns and trains significantly faster than the LSTM-based alternative while achieving competitive anomaly detection performance.

**Protocol (same as the paper)**

1. Train **only on normal** IDV(0) runs.
2. Z-score sensors with mean/std from the training runs only.
3. Cut **sliding windows inside each simulation run** (no mixing across runs).
4. Anomaly score = mean squared reconstruction error. Flag a window if error > a percentile of *training* errors (paper: 98.65th).
5. Exclude IDV 3, 9, 15.

**What you get that the paper reports at window level, plus run-level FPR/TPR**

The paper scores **windows**. Operators care about **runs**. A 48-hour healthy run has ~900 windows, so “any window above the 98.65th percentile” is almost guaranteed to fire (high FPR). This notebook therefore reports both:

| | paper (window) | this notebook (run) |
|---|---|---|
| score | window MSE | max window MSE on the trajectory (post-onset if the run is labelled faulty) |
| threshold | 98.65th percentile of *train* window errors | mean + 3 sd of *val* healthy **run** scores |
| **FPR** | share of healthy windows | share of healthy **runs** we call faulty |
| **TPR** | share of post-onset windows | share of faulty **runs** we call faulty |
| prefix FAR | — | share of faulty runs that already look faulty *before* onset |

This notebook is a **small sample** so it trains in seconds. Bump the counts at the top when you want the paper-scale experiment.


In [1]:
from __future__ import annotations

import logging
import time
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    auc,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

log = logging.getLogger("conv_ae")
if not log.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(asctime)s  %(levelname)-7s  %(message)s", datefmt="%H:%M:%S"))
    log.addHandler(handler)
    log.setLevel(logging.INFO)
    log.propagate = False

CSV_PATH = Path("te_process.csv")
CACHE_PATH = Path("conv_ae_small_sample.csv")  # gitignored via *.csv
CHUNKSIZE = 250_000
META_COLS = ["faultNumber", "simulationRun", "sample", "source", "fault_status"]

# Paper Table 4 (Conv-AE). Keep these; only the *number of runs* is reduced.
SEQ_LEN = 40
CONV_FILTERS = 64
KERNEL_SIZE = 3
LATENT_FILTERS = 128
DROPOUT = 0.119
LEARNING_RATE = 0.00853
BATCH_SIZE = 128          # paper 128; smaller batch is fine on a tiny sample
THRESHOLD_PERCENTILE = 98.65
USE_TRANSFORMER = False  # paper: best without a latent Transformer
STRIDED = True           # paper text: strided encoder + transposed decoder
PERSIST_K = 10           # consecutive alarming windows to call a run faulty
RUN_SCORE_K_SD = 3.0     # run threshold = val healthy max-MSE mean + k sd
PERSIST_K = 10           # consecutive alarming windows to call a run faulty
RUN_SCORE_K_SD = 3.0     # run threshold = val healthy max-MSE mean + k sd

# Small-sample teaching (paper used 100/50 train/val normal runs).
SOURCE = "test"          # paper test protocol: 960 samples, fault at sample 161
N_TRAIN_NORMAL = 100
N_VAL_NORMAL = 50
N_TEST_NORMAL = 120
N_FAULTY_PER_IDV = 50
EXCLUDE_FAULTS = {3, 9, 15}
INCLUDED_FAULTS = tuple(i for i in range(1, 21) if i not in EXCLUDE_FAULTS)

MAX_EPOCHS = 15
PATIENCE = 4
SPLIT_SEED = 42
TRAIN_FAULT_SAMPLE = 21   # Rieth train files: disturbance at 1 h
TEST_FAULT_SAMPLE = 161   # Rieth test files: disturbance at 8 h
SAMPLE_MINUTES = 3

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
torch.manual_seed(SPLIT_SEED)
np.random.seed(SPLIT_SEED)

log.info("csv        %s", CSV_PATH.resolve())
log.info("exists     %s", CSV_PATH.exists())
log.info("device     %s", device)
log.info("source     %s  seq_len=%s  strided=%s", SOURCE, SEQ_LEN, STRIDED)
log.info(
    "sample     train=%s val=%s test_normal=%s  faulty=%s/idv  idvs=%s",
    N_TRAIN_NORMAL, N_VAL_NORMAL, N_TEST_NORMAL, N_FAULTY_PER_IDV, len(INCLUDED_FAULTS),
)


12:53:05  INFO     csv        /Users/ob/Projects/hackathon_csv/data_processing/te_process.csv
12:53:05  INFO     exists     True
12:53:05  INFO     device     mps
12:53:05  INFO     source     test  seq_len=40  strided=True
12:53:05  INFO     sample     train=100 val=50 test_normal=120  faulty=50/idv  idvs=17


## 1. Load a small run sample

`te_process.csv` is 15.3M rows. We **do not** train on that. Stream it once, keep a quota of complete simulation runs, cache them, and stop.

IDV(0) runs are later shuffled into train / val / held-out test (whole trajectories, never cut in half). Faulty runs are evaluation-only.


In [2]:
def sensor_columns(columns) -> list[str]:
    return [c for c in columns if str(c).startswith("xmeas_") or str(c).startswith("xmv_")]


def compact_frame(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["faultNumber"] = out["faultNumber"].astype(np.int16)
    out["simulationRun"] = out["simulationRun"].astype(np.int16)
    out["sample"] = out["sample"].astype(np.int16)
    out["source"] = out["source"].astype("category")
    if "fault_status" in out.columns:
        out["fault_status"] = out["fault_status"].astype("category")
    for c in sensor_columns(out.columns):
        out[c] = pd.to_numeric(out[c], downcast="float")
    return out


def fault_onset(source: str | pd.Series) -> np.ndarray | int:
    if isinstance(source, str):
        return TEST_FAULT_SAMPLE if source == "test" else TRAIN_FAULT_SAMPLE
    return np.where(np.asarray(source) == "test", TEST_FAULT_SAMPLE, TRAIN_FAULT_SAMPLE)


def wanted_fault(fault: int) -> bool:
    return fault == 0 or fault in INCLUDED_FAULTS


def load_or_cache_sample(path: Path = CSV_PATH, cache: Path = CACHE_PATH) -> pd.DataFrame:
    if cache.exists():
        df = compact_frame(pd.read_csv(cache))
        log.info("cache hit  %s  rows=%s  runs=%s", cache, f"{len(df):,}", df.groupby(["source", "faultNumber", "simulationRun"]).ngroups)
        return df

    header = pd.read_csv(path, nrows=0)
    features = sensor_columns(header.columns)
    usecols = [c for c in META_COLS + features if c in header.columns]

    n_normal_needed = N_TRAIN_NORMAL + N_VAL_NORMAL + N_TEST_NORMAL
    quotas = {(SOURCE, 0): n_normal_needed}
    for fault in INCLUDED_FAULTS:
        quotas[(SOURCE, fault)] = N_FAULTY_PER_IDV

    done: dict[tuple[str, int], set[int]] = defaultdict(set)
    buffers: dict[tuple[str, int, int], list[pd.DataFrame]] = {}
    kept: list[pd.DataFrame] = []
    n_seen = 0
    t0 = time.perf_counter()

    def quotas_full() -> bool:
        return all(len(done[(src, f)]) >= n for (src, f), n in quotas.items())

    def flush_run(key: tuple[str, int, int]) -> None:
        src, fault, run = key
        quota_key = (src, int(fault))
        if quota_key not in quotas or len(done[quota_key]) >= quotas[quota_key]:
            buffers.pop(key, None)
            return
        parts = buffers.pop(key, None)
        if not parts:
            return
        run_df = pd.concat(parts, ignore_index=True)
        kept.append(run_df)
        done[quota_key].add(int(run))

    for i, chunk in enumerate(pd.read_csv(path, usecols=usecols, chunksize=CHUNKSIZE), start=1):
        n_seen += len(chunk)
        chunk = chunk.loc[chunk["source"].astype(str).eq(SOURCE)]
        if chunk.empty:
            if i == 1 or i % 8 == 0:
                log.info("chunk %03d  scanned=%s  kept_runs=%s  elapsed=%.1fs", i, f"{n_seen:,}", sum(len(v) for v in done.values()), time.perf_counter() - t0)
            continue

        chunk = compact_frame(chunk)
        live = chunk.loc[chunk["faultNumber"].map(wanted_fault)]
        if live.empty:
            if quotas_full():
                log.info("chunk %03d  quotas filled  scanned=%s", i, f"{n_seen:,}")
                break
            continue

        for (src, fault, run), g in live.groupby(["source", "faultNumber", "simulationRun"], sort=False, observed=True):
            src_s, fault_i, run_i = str(src), int(fault), int(run)
            quota_key = (src_s, fault_i)
            if quota_key not in quotas or len(done[quota_key]) >= quotas[quota_key]:
                continue
            key = (src_s, fault_i, run_i)
            buffers.setdefault(key, []).append(g)
            expected = 960 if src_s == "test" else 500
            n_have = sum(len(p) for p in buffers[key])
            if n_have >= expected:
                flush_run(key)

        if i == 1 or i % 8 == 0:
            log.info(
                "chunk %03d  scanned=%s  kept_runs=%s  buffered=%s  elapsed=%.1fs",
                i, f"{n_seen:,}", sum(len(v) for v in done.values()), len(buffers), time.perf_counter() - t0,
            )
        if quotas_full() and not buffers:
            break

    for key in list(buffers):
        flush_run(key)

    if not kept:
        raise RuntimeError(f"no sample rows collected from {path}")

    out = (
        pd.concat(kept, ignore_index=True)
        .sort_values(["source", "faultNumber", "simulationRun", "sample"], kind="mergesort")
        .reset_index(drop=True)
    )
    out.to_csv(cache, index=False)
    log.info(
        "cache write %s  rows=%s  runs=%s  scanned=%s  elapsed=%.1fs",
        cache, f"{len(out):,}", out.groupby(["source", "faultNumber", "simulationRun"]).ngroups, f"{n_seen:,}", time.perf_counter() - t0,
    )
    return out


df = load_or_cache_sample()
FEATURE_COLS = sensor_columns(df.columns)

summary = (
    df.groupby(["source", "faultNumber"], observed=True)
    .agg(n=("sample", "size"), runs=("simulationRun", "nunique"), smin=("sample", "min"), smax=("sample", "max"))
)
print("loaded sample")
display(summary)
print(f"rows={len(df):,}  sensors={len(FEATURE_COLS)}  faults={sorted(df.loc[df.faultNumber.gt(0), 'faultNumber'].unique())}")


loaded sample
                         n  runs  smin  smax
source faultNumber                          
test   0            259200   270     1   960
       1             48000    50     1   960
       2             48000    50     1   960
       4             48000    50     1   960
       5             48000    50     1   960
       6             48000    50     1   960
       7             48000    50     1   960
       8             48000    50     1   960
       10            48000    50     1   960
       11            48000    50     1   960
       12            48000    50     1   960
       13            48000    50     1   960
       14            48000    50     1   960
       16            48000    50     1   960
       17            48000    50     1   960
       18            48000    50     1   960
       19            48000    50     1   960
       20            48000    50     1   960
rows=1,075,200  sensors=52  faults=[np.int16(1), np.int16(2), np.int16(4), np.int16(5)

## 2. Split normal runs, scale, window

Paper: windows of length \(w=40\) (about 2 hours at 3 min/sample), stride 1, **inside each run**. The window label is the fault state at the **last** time step.

Faulty test runs are still healthy until sample 161. Do not use CSV `fault_status` as a sample label.


In [3]:
def split_normal_runs(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    normal = frame.loc[frame["faultNumber"].eq(0)]
    keys = normal[["source", "simulationRun"]].drop_duplicates().reset_index(drop=True)
    rng = np.random.default_rng(SPLIT_SEED)
    order = rng.permutation(len(keys))
    n_train, n_val = N_TRAIN_NORMAL, N_VAL_NORMAL
    if n_train + n_val + N_TEST_NORMAL > len(keys):
        raise RuntimeError(f"need {n_train + n_val + N_TEST_NORMAL} IDV(0) runs, have {len(keys)}")
    train_keys = keys.iloc[order[:n_train]]
    val_keys = keys.iloc[order[n_train:n_train + n_val]]
    test_keys = keys.iloc[order[n_train + n_val:n_train + n_val + N_TEST_NORMAL]]

    def take(k: pd.DataFrame) -> pd.DataFrame:
        return normal.merge(k, on=["source", "simulationRun"], how="inner")

    train, val, test = take(train_keys), take(val_keys), take(test_keys)
    log.info("normal split  train_runs=%s  val_runs=%s  test_runs=%s", n_train, n_val, N_TEST_NORMAL)
    return train, val, test


train_normal, val_normal, test_normal = split_normal_runs(df)
faulty = df.loc[df["faultNumber"].gt(0)].copy()

scaler = StandardScaler()
scaler.fit(train_normal[FEATURE_COLS].to_numpy(dtype=np.float32))
log.info("scaler fit on %s train-normal rows  (%s runs)", f"{len(train_normal):,}", train_normal["simulationRun"].nunique())


def scaled_run_matrix(run: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    ordered = run.sort_values("sample", kind="mergesort")
    X = scaler.transform(ordered[FEATURE_COLS].to_numpy(dtype=np.float32)).astype(np.float32)
    samples = ordered["sample"].to_numpy(dtype=np.int32)
    return X, samples


def windows_from_run(run: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return (windows, end_sample, y_true_at_end). y_true uses onset, not fault_status."""
    X, samples = scaled_run_matrix(run)
    if len(X) < SEQ_LEN:
        return (
            np.empty((0, SEQ_LEN, X.shape[1]), dtype=np.float32),
            np.empty((0,), dtype=np.int32),
            np.empty((0,), dtype=np.int8),
        )
    n = len(X) - SEQ_LEN + 1
    # sliding windows: (n, SEQ_LEN, n_features)
    idx = np.arange(SEQ_LEN)[None, :] + np.arange(n)[:, None]
    windows = X[idx]
    end_sample = samples[SEQ_LEN - 1:]
    src = str(run["source"].iloc[0])
    fault = int(run["faultNumber"].iloc[0])
    onset = fault_onset(src)
    y = ((fault > 0) & (end_sample >= onset)).astype(np.int8)
    return windows, end_sample, y


def stack_windows(frame: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, np.ndarray, pd.DataFrame]:
    xs, ends, ys, meta = [], [], [], []
    for (src, fault, run), g in frame.groupby(["source", "faultNumber", "simulationRun"], sort=False, observed=True):
        w, end, y = windows_from_run(g)
        if len(w) == 0:
            continue
        xs.append(w)
        ends.append(end)
        ys.append(y)
        meta.append(
            pd.DataFrame(
                {
                    "source": str(src),
                    "faultNumber": int(fault),
                    "simulationRun": int(run),
                    "end_sample": end,
                    "y_true": y,
                }
            )
        )
    if not xs:
        raise RuntimeError("no windows")
    return np.concatenate(xs), np.concatenate(ends), np.concatenate(ys), pd.concat(meta, ignore_index=True)


X_train, _, _, meta_train = stack_windows(train_normal)
X_val, _, _, meta_val = stack_windows(val_normal)
log.info("windows  train=%s  val=%s  shape=%s", f"{len(X_train):,}", f"{len(X_val):,}", X_train.shape)


## 3. Conv-AE

Encoder: two strided `Conv1d` layers (40 → 20 → 10 time steps, channels 52 → 64 → 128).  
Decoder: matching `ConvTranspose1d` back to `(batch, 40, 52)`.

Trained with MSE reconstruction loss on **normal windows only**.


In [4]:
class ConvAutoencoder(nn.Module):
    """1D conv AE. STRIDED=True matches the paper paragraph; False is the same-length official notebook."""

    def __init__(
        self,
        n_features: int,
        conv_filters: int = CONV_FILTERS,
        kernel_size: int = KERNEL_SIZE,
        latent_filters: int = LATENT_FILTERS,
        dropout: float = DROPOUT,
        strided: bool = STRIDED,
    ):
        super().__init__()
        pad = kernel_size // 2
        if strided:
            self.encoder = nn.Sequential(
                nn.Conv1d(n_features, conv_filters, kernel_size, stride=2, padding=pad),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Conv1d(conv_filters, latent_filters, kernel_size, stride=2, padding=pad),
                nn.ReLU(),
            )
            self.decoder = nn.Sequential(
                nn.ConvTranspose1d(latent_filters, conv_filters, kernel_size, stride=2, padding=pad, output_padding=1),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.ConvTranspose1d(conv_filters, n_features, kernel_size, stride=2, padding=pad, output_padding=1),
            )
        else:
            self.encoder = nn.Sequential(
                nn.Conv1d(n_features, conv_filters, kernel_size, padding=pad),
                nn.ReLU(),
                nn.Conv1d(conv_filters, latent_filters, kernel_size, padding=pad),
                nn.ReLU(),
            )
            self.decoder = nn.Sequential(
                nn.Conv1d(latent_filters, conv_filters, kernel_size, padding=pad),
                nn.ReLU(),
                nn.Conv1d(conv_filters, n_features, kernel_size, padding=pad),
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, time, sensors) -> conv wants (batch, sensors, time)
        z = self.encoder(x.transpose(1, 2))
        rec = self.decoder(z)
        return rec.transpose(1, 2)


def make_loader(X: np.ndarray, shuffle: bool) -> DataLoader:
    ds = TensorDataset(torch.from_numpy(X))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, drop_last=False)


model = ConvAutoencoder(n_features=len(FEATURE_COLS)).to(device)
n_params = sum(p.numel() for p in model.parameters())
log.info("model  params=%s  %s", f"{n_params:,}", model)

# shape check
with torch.no_grad():
    probe = torch.from_numpy(X_train[:2]).to(device)
    rec = model(probe)
    assert rec.shape == probe.shape, (rec.shape, probe.shape)
    log.info("forward ok  %s -> %s", tuple(probe.shape), tuple(rec.shape))


In [5]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
train_loader = make_loader(X_train, shuffle=True)
val_loader = make_loader(X_val, shuffle=False)

history = {"epoch": [], "train_loss": [], "val_loss": []}
best_val = float("inf")
best_state = None
stall = 0
t0 = time.perf_counter()

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    train_loss = 0.0
    n_seen = 0
    for (xb,) in train_loader:
        xb = xb.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(xb), xb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(xb)
        n_seen += len(xb)
    train_loss /= max(n_seen, 1)

    model.eval()
    val_loss = 0.0
    n_seen = 0
    with torch.no_grad():
        for (xb,) in val_loader:
            xb = xb.to(device)
            loss = criterion(model(xb), xb)
            val_loss += loss.item() * len(xb)
            n_seen += len(xb)
    val_loss /= max(n_seen, 1)

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    log.info("epoch %02d/%s  train=%.6f  val=%.6f", epoch, MAX_EPOCHS, train_loss, val_loss)

    if val_loss + 1e-8 < best_val:
        best_val = val_loss
        stall = 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        stall += 1
        if stall >= PATIENCE:
            log.info("early stop at epoch %s", epoch)
            break

if best_state is not None:
    model.load_state_dict(best_state)
log.info("train done  %.1fs  best_val=%.6f", time.perf_counter() - t0, best_val)

fig, ax = plt.subplots(figsize=(7.5, 3.2))
ax.plot(history["epoch"], history["train_loss"], label="train MSE")
ax.plot(history["epoch"], history["val_loss"], label="val MSE")
ax.set_xlabel("epoch")
ax.set_ylabel("reconstruction MSE")
ax.set_title("Conv-AE training (normal windows only)")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


<cell5>:65: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## 4. Threshold from normal training errors

Paper: flag a window as anomalous if its reconstruction MSE is above the **98.65th percentile** of training errors. High threshold → fewer false alarms, more missed subtle faults.


In [6]:
@torch.no_grad()
def reconstruction_errors(X: np.ndarray, bs: int = 256) -> np.ndarray:
    model.eval()
    out = np.empty(len(X), dtype=np.float64)
    i = 0
    while i < len(X):
        xb = torch.from_numpy(X[i : i + bs]).to(device)
        rec = model(xb)
        mse = ((rec - xb) ** 2).mean(dim=(1, 2)).cpu().numpy()
        out[i : i + len(mse)] = mse
        i += bs
    return out


train_errors = reconstruction_errors(X_train)
val_errors = reconstruction_errors(X_val)
threshold = float(np.percentile(train_errors, THRESHOLD_PERCENTILE))
log.info(
    "threshold  p=%.2f  value=%.6f  train_mean=%.6f  val_mean=%.6f  val_FPR_windows=%.3f",
    THRESHOLD_PERCENTILE,
    threshold,
    train_errors.mean(),
    val_errors.mean(),
    (val_errors > threshold).mean(),
)


## 5. Score every held-out run (healthy + faulty)

Each run gets a reconstruction-error series. We then try three ways to say “this run is faulty”:

| rule | fire if |
|---|---|
| `any_window` | any window MSE > paper window threshold (98.65th of train) |
| `persist_k` | ≥ `PERSIST_K` **consecutive** windows over that threshold |
| `run_max` | max window MSE > val-healthy run-score mean + 3 sd |

`any_window` is what you get if you naively promote the paper’s window rule to a whole trajectory — long healthy runs almost always have one spike. `run_max` is the operator rule: one score per run, threshold fit on **val IDV(0) only**.

- **FPR** = share of held-out healthy runs we call faulty
- **TPR** = share of faulty runs we call faulty using **post-onset** windows
- **prefix FAR** = share of faulty runs that already look faulty *before* onset


In [7]:
def iter_runs(frame: pd.DataFrame):
    for key, g in frame.groupby(["source", "faultNumber", "simulationRun"], sort=False, observed=True):
        yield key, g


def longest_true_run(mask: np.ndarray) -> int:
    best = cur = 0
    for v in mask:
        cur = cur + 1 if v else 0
        best = max(best, cur)
    return int(best)


def score_run(run: pd.DataFrame) -> dict:
    windows, end_sample, y_true = windows_from_run(run)
    src = str(run["source"].iloc[0])
    fault = int(run["faultNumber"].iloc[0])
    sim = int(run["simulationRun"].iloc[0])
    onset = int(fault_onset(src))
    rec = {
        "source": src,
        "faultNumber": fault,
        "simulationRun": sim,
        "n_windows": int(len(windows)),
        "y_run": int(fault > 0),
        "onset": onset,
    }
    empty = {
        "max_err": np.nan, "max_err_post": np.nan, "max_err_pre": np.nan,
        "n_alarm": 0, "persist_pre": 0, "persist_post": 0, "persist_all": 0,
    }
    if len(windows) == 0:
        rec.update(empty)
        return rec

    errors = reconstruction_errors(windows)
    pre = end_sample < onset
    post = end_sample >= onset if fault > 0 else np.ones(len(errors), dtype=bool)
    alarm = errors > threshold
    rec["max_err"] = float(errors.max())
    rec["max_err_pre"] = float(errors[pre].max()) if pre.any() else np.nan
    rec["max_err_post"] = float(errors[post].max()) if post.any() else float(errors.max())
    rec["n_alarm"] = int(alarm.sum())
    rec["persist_all"] = longest_true_run(alarm)
    rec["persist_pre"] = longest_true_run(alarm & pre) if pre.any() else 0
    rec["persist_post"] = longest_true_run(alarm & post)
    rec["end_sample"] = end_sample
    rec["errors"] = errors
    rec["y_true"] = y_true
    return rec


eval_frames = {
    "healthy_test": test_normal,
    "healthy_val": val_normal,
    "faulty": faulty,
}

run_rows = []
example = None
EXAMPLE_KEY = (SOURCE, 1)  # first included easy fault, any run

for split_name, frame in eval_frames.items():
    n = 0
    for key, run in iter_runs(frame):
        scored = score_run(run)
        scored["split"] = split_name
        run_rows.append({k: scored[k] for k in scored if k not in {"end_sample", "errors", "y_true"}})
        n += 1
        if (
            example is None
            and scored["faultNumber"] == EXAMPLE_KEY[1]
            and scored["source"] == EXAMPLE_KEY[0]
        ):
            example = scored
    log.info("scored %s  runs=%s", split_name, n)

runs = pd.DataFrame(run_rows)
display(runs.drop(columns=["source"]).head())


   faultNumber  simulationRun  n_windows  y_run  onset   max_err  max_err_pre  max_err_post  n_alarm  persist_all  persist_pre  persist_post         split
0            0              2        921      0    161  0.285845     0.277127      0.285845        0            0            0             0  healthy_test
1            0              6        921      0    161  0.306331     0.292104      0.306331       10            1            0             1  healthy_test
2            0              7        921      0    161  0.316241     0.281335      0.316241       11            5            0             5  healthy_test
3            0             11        921      0    161  0.300729     0.271094      0.300729        7            1            0             1  healthy_test
4            0             12        921      0    161  0.289546     0.276947      0.289546        0            0            0             0  healthy_test


In [8]:
def rate(mask: pd.Series) -> float:
    return float(mask.mean()) if len(mask) else float("nan")


healthy = runs.loc[runs["y_run"].eq(0)].copy()
fault_runs = runs.loc[runs["y_run"].eq(1)].copy()
h_val = healthy.loc[healthy["split"].eq("healthy_val")]
h_test = healthy.loc[healthy["split"].eq("healthy_test")]

val_mu = float(h_val["max_err"].mean())
val_sd = float(h_val["max_err"].std(ddof=0) or 1e-6)
run_threshold = val_mu + RUN_SCORE_K_SD * val_sd
log.info("run threshold  val_mean=%.4f  val_sd=%.4f  mean+%ssd=%.4f", val_mu, val_sd, int(RUN_SCORE_K_SD), run_threshold)


def apply_rule(frame: pd.DataFrame, rule: str, post: bool) -> pd.Series:
    if rule == "any_window":
        col = "max_err_post" if post else "max_err_pre"
        return frame[col] > threshold
    if rule == "persist_k":
        col = "persist_post" if post else "persist_pre"
        return frame[col] >= PERSIST_K
    if rule == "run_max":
        col = "max_err_post" if post else "max_err_pre"
        return frame[col] > run_threshold
    raise ValueError(rule)


rows = []
cms = {}
for rule in ("any_window", "persist_k", "run_max"):
    pred_h_test = apply_rule(h_test, rule, post=True)
    pred_h_val = apply_rule(h_val, rule, post=True)
    pred_fault = apply_rule(fault_runs, rule, post=True)
    pred_prefix = apply_rule(fault_runs, rule, post=False)
    y_true = np.r_[np.zeros(len(h_test), dtype=int), np.ones(len(fault_runs), dtype=int)]
    y_pred = np.r_[pred_h_test.astype(int), pred_fault.astype(int)]
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    tpr = rate(pred_fault)
    fpr_test = rate(pred_h_test)
    rows.append(
        {
            "rule": rule,
            "TPR": tpr,
            "FPR_test": fpr_test,
            "FPR_val": rate(pred_h_val),
            "prefix_FAR": rate(pred_prefix),
            "Youden": tpr - fpr_test,
            "TP": int(tp),
            "FN": int(fn),
            "FP": int(fp),
            "TN": int(tn),
        }
    )
    cms[rule] = cm
    if rule == "run_max":
        fault_runs = fault_runs.assign(pred=pred_fault.astype(int), prefix_fire=pred_prefix.astype(int))
        h_test = h_test.assign(pred=pred_h_test.astype(int))

sheet = pd.DataFrame(rows).set_index("rule")
print("run-level detection  (healthy test runs=%s, faulty runs=%s)" % (len(h_test), len(fault_runs)))
display(sheet.round(4))

print("\nconfusion for run_max  rows=true [healthy, faulty]  cols=pred [healthy, faulty]")
print(pd.DataFrame(cms["run_max"], index=["true healthy", "true faulty"], columns=["pred healthy", "pred faulty"]))

by_fault = (
    fault_runs.assign(pred=apply_rule(fault_runs, "run_max", post=True), prefix_fire=apply_rule(fault_runs, "run_max", post=False))
    .groupby("faultNumber")
    .agg(runs=("simulationRun", "nunique"), TPR=("pred", "mean"), prefix_FAR=("prefix_fire", "mean"), median_max_err=("max_err_post", "median"))
    .sort_index()
)
print("\nrun_max TPR by IDV")
display(by_fault.round(3))


run-level detection  (healthy test runs=120, faulty runs=850)
            TPR  FPR_test  FPR_val  prefix_FAR  Youden   TP  FN  FP   TN
rule                                                                    
any_window  1.0    0.8167     0.84        0.12  0.1833  850   0  98   22
persist_k   1.0    0.1333     0.10        0.00  0.8667  850   0  16  104
run_max     1.0    0.0083     0.00        0.00  0.9917  850   0   1  119

confusion for run_max  rows=true [healthy, faulty]  cols=pred [healthy, faulty]
              pred healthy  pred faulty
true healthy           119            1
true faulty              0          850

run_max TPR by IDV
             runs  TPR  prefix_FAR  median_max_err
faultNumber                                       
1              50  1.0         0.0           5.387
2              50  1.0         0.0          16.026
4              50  1.0         0.0           1.161
5              50  1.0         0.0           1.020
6              50  1.0         0.0         395

Window-level metrics match the paper's binary table (accuracy / F1 / ROC-AUC). They treat every sliding window as a sample. Run-level FPR/TPR above is the operator question: *is this trajectory faulty?*


In [9]:
# Window-level scores on held-out healthy test + faulty runs (post-onset vs healthy).
X_eval, _, y_eval, meta_eval = stack_windows(pd.concat([test_normal, faulty], ignore_index=True))
err_eval = reconstruction_errors(X_eval)
y_hat = (err_eval > threshold).astype(int)

# Drop windows that sit in a faulty run *before* onset: they are healthy physiology
# but the paper's binary_test labels used faultNumber of the run. We keep both views.
win = meta_eval.copy()
win["error"] = err_eval
win["y_hat"] = y_hat
# strictly: y_true already uses onset
y = win["y_true"].to_numpy()
y_hat_w = win["y_hat"].to_numpy()
scores = win["error"].to_numpy()

roc = roc_auc_score(y, scores) if y.min() != y.max() else float("nan")
prec_c, rec_c, _ = precision_recall_curve(y, scores)
pr = auc(rec_c, prec_c)
window_metrics = pd.Series(
    {
        "n_windows": len(win),
        "accuracy": accuracy_score(y, y_hat_w),
        "precision_fault": precision_score(y, y_hat_w, zero_division=0),
        "recall_TPR": recall_score(y, y_hat_w, zero_division=0),
        "FPR": ((y_hat_w == 1) & (y == 0)).sum() / max((y == 0).sum(), 1),
        "f1_fault": f1_score(y, y_hat_w, zero_division=0),
        "roc_auc": roc,
        "pr_auc": pr,
    }
)
print("window-level (onset-aware labels)")
display(window_metrics.to_frame("conv_ae").T.round(4))


window-level (onset-aware labels)
         n_windows  accuracy  precision_fault  recall_TPR     FPR  f1_fault  roc_auc  pr_auc
conv_ae   893370.0    0.9697           0.9969      0.9632  0.0097    0.9797   0.9889   0.997


In [10]:
fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.4))

healthy_err = h_test["max_err"].dropna()
fault_err = fault_runs["max_err_post"].dropna()
axes[0].hist(healthy_err, bins=12, alpha=0.65, label="healthy test (max MSE)", color="C0", density=True)
axes[0].hist(fault_err, bins=12, alpha=0.65, label="faulty (max post MSE)", color="C3", density=True)
axes[0].axvline(threshold, color="0.4", ls=":", lw=1.2, label=f"window p{THRESHOLD_PERCENTILE:.2f}")
axes[0].axvline(run_threshold, color="k", ls="--", lw=1.2, label=f"run mean+{RUN_SCORE_K_SD:.0f}sd")
axes[0].set_xlabel("run score (max window MSE)")
axes[0].set_ylabel("density")
axes[0].set_title("Run scores")
axes[0].legend(fontsize=8)

fpr_c, tpr_c, _ = roc_curve(
    np.r_[np.zeros(len(healthy_err)), np.ones(len(fault_err))],
    np.r_[healthy_err.to_numpy(), fault_err.to_numpy()],
)
axes[1].plot(fpr_c, tpr_c, lw=2, label=f"run ROC  AUC={auc(fpr_c, tpr_c):.3f}")
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.4)
axes[1].set_xlabel("FPR")
axes[1].set_ylabel("TPR")
axes[1].set_title("Run-level ROC")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)

if example is None:
    axes[2].set_visible(False)
else:
    hours = (example["end_sample"] - 1) * SAMPLE_MINUTES / 60
    onset_h = (example["onset"] - 1) * SAMPLE_MINUTES / 60
    axes[2].plot(hours, example["errors"], lw=1.2, color="C0")
    axes[2].axhline(threshold, color="0.4", ls=":", lw=1, label="window")
    axes[2].axhline(run_threshold, color="k", ls="--", lw=1, label="run")
    axes[2].axvline(onset_h, color="C3", ls=":", lw=1.4, label="onset")
    axes[2].set_xlabel("hours")
    axes[2].set_ylabel("window MSE")
    axes[2].set_title(f"example IDV({example['faultNumber']}) run {example['simulationRun']}")
    axes[2].legend(fontsize=8)
    axes[2].set_xlim(0, float(hours.max()) if len(hours) else 1)

fig.tight_layout()
plt.show()

rm = sheet.loc["run_max"]
print(
    f"run_max  healthy test={len(h_test)}  FPR={rm['FPR_test']:.1%}   |   "
    f"faulty={len(fault_runs)}  TPR={rm['TPR']:.1%}  prefix FAR={rm['prefix_FAR']:.1%}"
)
if example is not None:
    fired = example["max_err_post"] > run_threshold
    print(f"example run predicted {'FAULTY' if fired else 'HEALTHY'}  (true = FAULTY)")


run_max  healthy test=120  FPR=0.8%   |   faulty=850  TPR=100.0%  prefix FAR=0.0%
example run predicted FAULTY  (true = FAULTY)


<cell10>:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## 6. Table 4 hyperparameters (official Conv-AE)

The cells above already use Table 4 numbers (`sequence_length=40`, `conv_filters=64`, `kernel_size=3`, `latent_filters=128`, `dropout=0.119`, `lr=0.00853`, `batch_size=128`, `threshold_percentile=98.65`) but with a **strided** encoder (the paper *text*). Table 4 was tuned on the supporting notebook: **same-length** `Conv1d` / `Conv1d` decoder and `use_transformer=False`.

This block retrains that official architecture on the **same** train/val/test runs and compares run-level FPR/TPR. Dropout is unused unless a Transformer is on — matching their code.

In [11]:
# Exact Table 4 Conv-AE (binary anomaly detection).
TABLE4 = {
    "sequence_length": 40,
    "conv_filters": 64,
    "kernel_size": 3,
    "latent_filters": 128,
    "use_transformer": False,
    "dropout": 0.119,
    "learning_rate": 0.00853,
    "batch_size": 128,
    "threshold_percentile": 98.65,
}

baseline_run = sheet.copy()
baseline_run.insert(0, "model", "strided (text)")
baseline_thr = float(threshold)
baseline_run_thr = float(run_threshold)
baseline_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

assert SEQ_LEN == TABLE4["sequence_length"], "windows were built with a different sequence_length"

model_t4 = ConvAutoencoder(
    n_features=len(FEATURE_COLS),
    conv_filters=TABLE4["conv_filters"],
    kernel_size=TABLE4["kernel_size"],
    latent_filters=TABLE4["latent_filters"],
    dropout=TABLE4["dropout"],
    strided=False,  # official notebook, not the paper paragraph
).to(device)
log.info("table4 model  params=%s  transformer=%s  strided=False", f"{sum(p.numel() for p in model_t4.parameters()):,}", TABLE4["use_transformer"])

criterion_t4 = nn.MSELoss()
opt_t4 = torch.optim.Adam(model_t4.parameters(), lr=TABLE4["learning_rate"])
loader_tr = DataLoader(TensorDataset(torch.from_numpy(X_train)), batch_size=TABLE4["batch_size"], shuffle=True)
loader_va = DataLoader(TensorDataset(torch.from_numpy(X_val)), batch_size=TABLE4["batch_size"], shuffle=False)

hist_t4 = {"epoch": [], "train_loss": [], "val_loss": []}
best_val_t4, best_state_t4, stall_t4 = float("inf"), None, 0
t0 = time.perf_counter()
for epoch in range(1, MAX_EPOCHS + 1):
    model_t4.train()
    tr_loss = n = 0.0
    for (xb,) in loader_tr:
        xb = xb.to(device)
        opt_t4.zero_grad(set_to_none=True)
        loss = criterion_t4(model_t4(xb), xb)
        loss.backward()
        opt_t4.step()
        tr_loss += loss.item() * len(xb)
        n += len(xb)
    tr_loss /= max(n, 1)

    model_t4.eval()
    va_loss = n = 0.0
    with torch.no_grad():
        for (xb,) in loader_va:
            xb = xb.to(device)
            loss = criterion_t4(model_t4(xb), xb)
            va_loss += loss.item() * len(xb)
            n += len(xb)
    va_loss /= max(n, 1)
    hist_t4["epoch"].append(epoch)
    hist_t4["train_loss"].append(tr_loss)
    hist_t4["val_loss"].append(va_loss)
    log.info("table4 epoch %02d/%s  train=%.6f  val=%.6f", epoch, MAX_EPOCHS, tr_loss, va_loss)
    if va_loss + 1e-8 < best_val_t4:
        best_val_t4, stall_t4 = va_loss, 0
        best_state_t4 = {k: v.detach().cpu().clone() for k, v in model_t4.state_dict().items()}
    else:
        stall_t4 += 1
        if stall_t4 >= PATIENCE:
            log.info("table4 early stop at epoch %s", epoch)
            break

if best_state_t4 is not None:
    model_t4.load_state_dict(best_state_t4)
log.info("table4 train done  %.1fs  best_val=%.6f", time.perf_counter() - t0, best_val_t4)

# Point the existing scorers at the Table 4 weights.
model = model_t4
train_errors_t4 = reconstruction_errors(X_train)
val_errors_t4 = reconstruction_errors(X_val)
threshold = float(np.percentile(train_errors_t4, TABLE4["threshold_percentile"]))
log.info(
    "table4 threshold  p=%.2f  value=%.6f  train_mean=%.6f  val_mean=%.6f  val_FPR_windows=%.3f",
    TABLE4["threshold_percentile"], threshold, train_errors_t4.mean(), val_errors_t4.mean(),
    (val_errors_t4 > threshold).mean(),
)

run_rows_t4 = []
example_t4 = None
for split_name, frame in eval_frames.items():
    n = 0
    for key, run in iter_runs(frame):
        scored = score_run(run)
        scored["split"] = split_name
        run_rows_t4.append({k: scored[k] for k in scored if k not in {"end_sample", "errors", "y_true"}})
        n += 1
        if example_t4 is None and scored["faultNumber"] == EXAMPLE_KEY[1] and scored["source"] == EXAMPLE_KEY[0]:
            example_t4 = scored
    log.info("table4 scored %s  runs=%s", split_name, n)

runs_t4 = pd.DataFrame(run_rows_t4)
healthy_t4 = runs_t4.loc[runs_t4["y_run"].eq(0)].copy()
fault_t4 = runs_t4.loc[runs_t4["y_run"].eq(1)].copy()
h_val_t4 = healthy_t4.loc[healthy_t4["split"].eq("healthy_val")]
h_test_t4 = healthy_t4.loc[healthy_t4["split"].eq("healthy_test")]
run_threshold_t4 = float(h_val_t4["max_err"].mean() + RUN_SCORE_K_SD * (h_val_t4["max_err"].std(ddof=0) or 1e-6))
log.info("table4 run threshold  mean+%ssd=%.4f", int(RUN_SCORE_K_SD), run_threshold_t4)

rows_t4 = []
for rule in ("any_window", "persist_k", "run_max"):
    if rule == "any_window":
        pred_h_test = h_test_t4["max_err"] > threshold
        pred_h_val = h_val_t4["max_err"] > threshold
        pred_fault = fault_t4["max_err_post"] > threshold
        pred_prefix = fault_t4["max_err_pre"] > threshold
    elif rule == "persist_k":
        pred_h_test = h_test_t4["persist_all"] >= PERSIST_K
        pred_h_val = h_val_t4["persist_all"] >= PERSIST_K
        pred_fault = fault_t4["persist_post"] >= PERSIST_K
        pred_prefix = fault_t4["persist_pre"] >= PERSIST_K
    else:
        pred_h_test = h_test_t4["max_err"] > run_threshold_t4
        pred_h_val = h_val_t4["max_err"] > run_threshold_t4
        pred_fault = fault_t4["max_err_post"] > run_threshold_t4
        pred_prefix = fault_t4["max_err_pre"] > run_threshold_t4
    y_true = np.r_[np.zeros(len(h_test_t4), dtype=int), np.ones(len(fault_t4), dtype=int)]
    y_pred = np.r_[pred_h_test.astype(int), pred_fault.astype(int)]
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    tpr = float(pred_fault.mean())
    fpr_test = float(pred_h_test.mean())
    rows_t4.append(
        {
            "model": "table4 official",
            "rule": rule,
            "TPR": tpr,
            "FPR_test": fpr_test,
            "FPR_val": float(pred_h_val.mean()),
            "prefix_FAR": float(pred_prefix.mean()),
            "Youden": tpr - fpr_test,
            "TP": int(tp), "FN": int(fn), "FP": int(fp), "TN": int(tn),
        }
    )

sheet_t4 = pd.DataFrame(rows_t4).set_index("rule")
compare = pd.concat([baseline_run.reset_index().assign(model="strided (text)"), sheet_t4.reset_index()])
compare = compare.set_index(["model", "rule"]).sort_index()
print("same runs, two architectures")
print(f"  strided window thr={baseline_thr:.4f}  run thr={baseline_run_thr:.4f}")
print(f"  table4  window thr={threshold:.4f}  run thr={run_threshold_t4:.4f}")
display(compare.round(4))

print("\nDid Table 4 improve run_max (TPR high, FPR low)?")
b = baseline_run.loc["run_max"]
t = sheet_t4.loc["run_max"]
print(f"  Youden  {b['Youden']:.4f} -> {t['Youden']:.4f}  ({t['Youden'] - b['Youden']:+.4f})")
print(f"  FPR     {b['FPR_test']:.4f} -> {t['FPR_test']:.4f}")
print(f"  TPR     {b['TPR']:.4f} -> {t['TPR']:.4f}")
print(f"  prefix  {b['prefix_FAR']:.4f} -> {t['prefix_FAR']:.4f}")

fig, ax = plt.subplots(figsize=(7.5, 3.2))
ax.plot(hist_t4["epoch"], hist_t4["train_loss"], label="table4 train")
ax.plot(hist_t4["epoch"], hist_t4["val_loss"], label="table4 val")
ax.set_xlabel("epoch")
ax.set_ylabel("reconstruction MSE")
ax.set_title("Table 4 official Conv-AE (same-length, no transformer)")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

# Restore strided weights if you keep working in later cells.
# model.load_state_dict(baseline_state); threshold = baseline_thr


same runs, two architectures
  strided window thr=0.2938  run thr=0.3395
  table4  window thr=0.0295  run thr=0.0404
                            TPR  FPR_test  FPR_val  prefix_FAR  Youden   TP  FN   FP   TN
model           rule                                                                     
strided (text)  any_window  1.0    0.8167     0.84        0.12  0.1833  850   0   98   22
                persist_k   1.0    0.1333     0.10        0.00  0.8667  850   0   16  104
                run_max     1.0    0.0083     0.00        0.00  0.9917  850   0    1  119
table4 official any_window  1.0    0.9667     1.00        0.22  0.0333  850   0  116    4
                persist_k   1.0    0.0333     0.02        0.00  0.9667  850   0    4  116
                run_max     1.0    0.0000     0.00        0.00  1.0000  850   0    0  120

Did Table 4 improve run_max (TPR high, FPR low)?
  Youden  0.9917 -> 1.0000  (+0.0083)
  FPR     0.0083 -> 0.0000
  TPR     1.0000 -> 1.0000
  prefix  0.0000 -> 0

<cell11>:171: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## Scale-up

This cell is documentation only. To follow the paper's binary split more closely, raise the quotas and re-run from the top after deleting `conv_ae_small_sample.csv`:

| | this notebook | paper (semi-supervised) |
|---|---:|---:|
| train normal runs | 8 | 100 |
| val normal runs | 4 | 50 |
| test normal runs | 8 | 120 |
| faulty runs / IDV | 2 | 50 |
| sequence length | 40 | 40 |
| threshold percentile | 98.65 | 98.65 |

Paper Conv-AE test numbers (full data): accuracy 99.39%, F1 0.9964, recall 99.70%, ROC-AUC 0.9988, trained in 56 s. On *new* simulator seeds the same threshold broke (accuracy 76%) because normal reconstruction error shifted — if you later add a new dataset, **recalibrate the percentile on fresh IDV(0)**.
